In [ ]:
Cover the cropland area with a 500m X 500m fishnet grid

In [ ]:
import geopandas as gpd
import rasterio
import numpy as np
from shapely.geometry import box
from pathlib import Path

# === 路径 ===
base_dir = Path("/Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_CGCS2000")
clcd_path = base_dir / "CLCD_v01_2017_albert_jiangsu_albers.tif"   # 用任一年即可定义范围
out_path  = base_dir / "Jiangsu_grid_500m_ID.gpkg"

# === Step 1. 读取 CLCD 的空间范围和投影 ===
with rasterio.open(clcd_path) as src:
    bounds = src.bounds
    crs = src.crs
    clcd_data = src.read(1)
    nodata = src.nodata if src.nodata is not None else 0
    transform = src.transform

print("CRS:", crs)
print("Bounds:", bounds)

# === Step 2. 设置格网分辨率（500 m） ===
grid_size = 500  # 单位: 米

xmin, ymin, xmax, ymax = bounds
cols = int(np.ceil((xmax - xmin) / grid_size))
rows = int(np.ceil((ymax - ymin) / grid_size))

# === Step 3. 生成所有格网（带 grid_id + 中心点坐标） ===
polygons, ids, x_centers, y_centers = [], [], [], []
gid = 1

for i in range(cols):
    for j in range(rows):
        x0 = xmin + i * grid_size
        y0 = ymin + j * grid_size
        x1 = x0 + grid_size
        y1 = y0 + grid_size
        geom = box(x0, y0, x1, y1)
        polygons.append(geom)
        ids.append(gid)
        x_centers.append((x0 + x1) / 2)
        y_centers.append((y0 + y1) / 2)
        gid += 1

fishnet = gpd.GeoDataFrame(
    {"grid_id": ids, "x_center": x_centers, "y_center": y_centers, "geometry": polygons},
    crs=crs
)
print(f"Created {len(fishnet):,} initial 500 m cells")

# === Step 4. 用 CLCD 判断哪些格网在有效陆地区域内 ===
# 通过检查格网中心点是否落在有效（非 0 / 非 NaN）像元上
def is_valid_cell(x, y):
    col, row = ~transform * (x, y)  
    row, col = int(row), int(col)



    if 0 <= row < clcd_data.shape[0] and 0 <= col < clcd_data.shape[1]:
        val = clcd_data[row, col]
        return np.isfinite(val) and val != nodata and val > 0
    return False

fishnet["is_valid"] = [
    is_valid_cell(x, y) for x, y in zip(fishnet.x_center, fishnet.y_center)
]
fishnet_valid = fishnet[fishnet["is_valid"]].drop(columns=["is_valid"])

print(f"✅ Valid grid cells retained: {len(fishnet_valid):,}")

# === Step 5. 保存 ===
fishnet_valid.to_file(out_path, driver="GPKG")
print(f"🎯 Saved to: {out_path}")



CRS: EPSG:4547
Bounds: BoundingBox(left=678838.6831488275, bottom=3411242.8532882654, right=1286760.032069308, top=3934231.8054719055)
Created 1,271,936 initial 500 m cells
✅ Valid grid cells retained: 410,620
🎯 Saved to: /Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_CGCS2000/Jiangsu_grid_500m_ID.gpkg


In [ ]:
import geopandas as gpd
import rasterio
import numpy as np
import pandas as pd
from rasterstats import zonal_stats
from pathlib import Path
from tqdm import tqdm

# === 路径 ===
base_dir = Path("/Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_CGCS2000")
grid_path = base_dir / "Jiangsu_grid_500m_ID.gpkg"
out_csv = base_dir / "Jiangsu_grid_cropland_2012_2022.csv"
out_gpkg = base_dir / "Jiangsu_grid_cropland_2012_2022.gpkg"

# === 参数 ===
years = range(2012, 2023)
cell_area = 500 * 500  # 每格面积，单位 m²

# === Step 1. 读取鱼网 ===
grid = gpd.read_file(grid_path)
print(f"✅ Loaded grid: {len(grid)} cells")

# === Step 2. 循环读取每年的 CLCD ===
all_years = []

for year in tqdm(years, desc="Processing CLCD yearly cropland area"):
    clcd_path = base_dir / f"CLCD_v01_{year}_albert_jiangsu_albers.tif"
    print(f"  → {year}: {clcd_path.name}")
    
    # zonal_stats: 计算每个polygon内值的比例
    stats = zonal_stats(
        vectors=grid,
        raster=str(clcd_path),
        categorical=True,
        nodata=0
    )
    
    # 将结果转为 DataFrame
    df = pd.DataFrame(stats).fillna(0)
    
    # 如果耕地代码为 1
    if 1 in df.columns:
        cropland_frac = df[1] / df.sum(axis=1)
        cropland_area = df[1] * (cell_area / df.sum(axis=1))
    else:
        cropland_frac = 0
        cropland_area = 0
    
    grid[f"cropland_area_{year}"] = cropland_area
    all_years.append(pd.DataFrame({
        "grid_id": grid["grid_id"],
        "year": year,
        "cropland_area_m2": cropland_area
    }))

# === Step 3. 导出 ===
# (a) Excel / CSV long format
df_out = pd.concat(all_years, ignore_index=True)
df_out.to_csv(out_csv, index=False)
print(f"💾 Saved yearly cropland CSV → {out_csv}")

# (b) GeoPackage with attributes
grid.to_file(out_gpkg, driver="GPKG")
print(f"🗺️ Saved GeoPackage → {out_gpkg}")


✅ Loaded grid: 410620 cells


Processing CLCD yearly cropland area:   0%|          | 0/11 [00:00<?, ?it/s]

  → 2012: CLCD_v01_2012_albert_jiangsu_albers.tif


Processing CLCD yearly cropland area:   9%|▉         | 1/11 [07:23<1:13:51, 443.15s/it]

  → 2013: CLCD_v01_2013_albert_jiangsu_albers.tif


...

  → 2021: CLCD_v01_2021_albert_jiangsu_albers.tif


Processing CLCD yearly cropland area:  91%|█████████ | 10/11 [1:13:06<07:15, 435.09s/it]

  → 2022: CLCD_v01_2022_albert_jiangsu_albers.tif


Processing CLCD yearly cropland area: 100%|██████████| 11/11 [1:20:23<00:00, 438.48s/it]


💾 Saved yearly cropland CSV → /Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_CGCS2000/Jiangsu_grid_cropland_2012_2022.csv
🗺️ Saved GeoPackage → /Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_CGCS2000/Jiangsu_grid_cropland_2012_2022.gpkg
